# NB11: UNI cancer-specific variants (Table 3)

Evaluates within-cancer UNI patient embeddings on TCGA val/test splits
using the manuscript's PCA -> Ridge -> Platt pipeline.

**Manuscript reference** (Methods + Table 3):
- UNI: ViT-L/16, 1,536-dimensional embeddings (Chen et al. 2024)
- Same PCA -> Ridge -> Platt pipeline as NB07
- UNI BRCA-specific: AUROC = 0.695 (delta = -0.071 vs OpenCLIP 0.766)
- UNI LUAD-specific: AUROC = 0.610 (delta = -0.156 vs OpenCLIP 0.766)
- Cancer-specific = restrict to one cancer's patients for both training
  (within the train split) and evaluation (within the val/test split)

UNI feature extraction uses the released UNI checkpoint and is performed
out-of-band; this notebook expects pre-computed patient-level UNI
embeddings on disk. To extract them, apply the UNI model to the same
tile coordinates produced by NB02, then mean-pool to the patient level.

**Required env**: `WORKSPACE`, `UNI_FEATURES_DIR` (directory containing
`uni_<cancer>_patient_embeddings.parquet`, with columns `patient` plus
1,536 feature columns).
**Outputs**: `results/uni/{table3_uni_variants.csv, report.json}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
UNI_FEATURES_DIR = Path(os.environ["UNI_FEATURES_DIR"]).resolve()
LABELS_DIR = WORKSPACE / "labels"
RESULTS_DIR = WORKSPACE / "results" / "uni"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked params (same as NB07)
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
HRD_THR = 33
BOOT_N = 200

# cancer-specific UNI variants reported in manuscript Table 3
COHORTS = {
    "BRCA": {
        "features": UNI_FEATURES_DIR / "uni_brca_patient_embeddings.parquet",
        "manuscript_AUROC": 0.695,
    },
    "LUAD": {
        "features": UNI_FEATURES_DIR / "uni_luad_patient_embeddings.parquet",
        "manuscript_AUROC": 0.610,
    },
}

print(f"UNI_FEATURES_DIR: {UNI_FEATURES_DIR}")
print(f"cohorts         : {list(COHORTS.keys())}")


In [ ]:
# load TCGA labels (NB04 + NB05) once
labels = pd.read_parquet(LABELS_DIR / "labels.parquet")
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")
labels = labels.loc[labels["HRD"].notna()].copy()
labels["HRD_top20"] = (labels["HRD"] >= HRD_THR).astype(int)
print(f"labelled patients: {len(labels):,}")


In [ ]:
# eval helper: same PCA -> Ridge -> Platt as NB07, restricted to one cancer's splits
def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        try:
            vals.append(fn(yb, pb))
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def evaluate_within_cancer(features_path, cancer, lbls):
    if not features_path.exists():
        return {"cancer": cancer, "error": f"missing features: {features_path}"}
    X = pd.read_parquet(features_path)
    if "patient" in X.columns:
        X = X.set_index("patient")
    X.index = X.index.astype(str).str.upper().str.slice(0, 12)

    sub = lbls.loc[lbls["cancer"] == cancer]
    common = sorted(set(X.index) & set(sub.index))
    if len(common) < 50:
        return {"cancer": cancer, "error": f"too few aligned patients ({len(common)})"}

    X = X.loc[common]
    sub = sub.loc[common].copy()
    idx_tr = sub.index[sub["split"] == "train"]
    idx_va = sub.index[sub["split"] == "val"]
    idx_te = sub.index[sub["split"] == "test"]

    if min(len(idx_tr), len(idx_va), len(idx_te)) < 5:
        return {"cancer": cancer, "error": "insufficient split sizes within cancer"}

    X_tr = X.loc[idx_tr].values.astype(np.float32)
    y_tr_cont = sub.loc[idx_tr, "HRD"].astype(float).values
    y_tr_bin = sub.loc[idx_tr, "HRD_top20"].astype(int).values

    pca_n = min(PCA_N, X_tr.shape[1] - 1, X_tr.shape[0] - 1)
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("pca", PCA(n_components=pca_n, random_state=SEED)),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
    ])
    pipe.fit(X_tr, y_tr_cont)

    z_tr = pipe.predict(X_tr).reshape(-1, 1)
    platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED).fit(z_tr, y_tr_bin)

    def predict(Xq):
        return platt.predict_proba(pipe.predict(Xq).reshape(-1, 1))[:, 1]

    out = {"cancer": cancer, "feat_dim": int(X.shape[1]),
           "n_train": int(len(idx_tr)), "n_val": int(len(idx_va)), "n_test": int(len(idx_te)),
           "pca_n_used": int(pca_n)}
    for tag, idx in [("val", idx_va), ("test", idx_te)]:
        Xq = X.loc[idx].values.astype(np.float32)
        y = sub.loc[idx, "HRD_top20"].astype(int).values
        if y.sum() == 0 or y.sum() == len(y):
            out[f"{tag}_AUROC"] = float("nan")
            continue
        p = predict(Xq)
        out[f"{tag}_AUROC"] = float(roc_auc_score(y, p))
        out[f"{tag}_AP"] = float(average_precision_score(y, p))
        out[f"{tag}_Brier"] = float(brier_score_loss(y, p))
        lo, hi = boot_ci(y, p, roc_auc_score)
        out[f"{tag}_AUROC_lo"] = lo
        out[f"{tag}_AUROC_hi"] = hi
    return out


In [ ]:
# run all cancer-specific UNI variants
rows = []
for cancer, cfg in COHORTS.items():
    print(f"\n--- UNI {cancer}-specific ---")
    res = evaluate_within_cancer(cfg["features"], cancer, labels)
    res["manuscript_AUROC"] = cfg["manuscript_AUROC"]
    if "error" in res:
        print(f"  {res['error']}")
    else:
        ms_ref = cfg["manuscript_AUROC"]
        delta = res.get("test_AUROC", float("nan")) - 0.766
        print(f"  test n={res['n_test']}  AUROC={res.get('test_AUROC', float('nan')):.3f}  "
              f"(ms ref {ms_ref:.3f})  delta_vs_OpenCLIP={delta:+.3f}")
    rows.append(res)

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / "table3_uni_variants.csv", index=False)
print()
print(df.to_string(index=False))


In [ ]:
# final report
report = {
    "seed": SEED,
    "params": {"PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA, "HRD_THR": HRD_THR, "BOOT_N": BOOT_N},
    "uni_features_dir": str(UNI_FEATURES_DIR),
    "results": rows,
    "manuscript_refs": {
        "UNI_BRCA_AUROC": 0.695,
        "UNI_BRCA_delta": -0.071,
        "UNI_LUAD_AUROC": 0.610,
        "UNI_LUAD_delta": -0.156,
        "OpenCLIP_test_AUROC": 0.766,
        "feat_dim": 1536,
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
